 Interactive Shakespeare Text Generation Interface
This notebook cell provides an interactive interface that allows users to generate Shakespearean-style text using fine-tuned BERT Generation Encoder-Decoder models.
The models are loaded directly from Hugging Face repositories:
https://huggingface.co/NahlaAboromi

How it works:
The user selects a dataset size (30k or 100K input-output pairs) and an epoch version (EPOCH1, EPOCH2, or EPOCH3) using dropdown menus.

The user provides an input prompt in the text area.

When the Generate Shakespearean Text button is clicked:
1. The appropriate model and tokenizer are loaded dynamically from Hugging Face.
2. The model generates a Shakespearean-style continuation for the input prompt.
3. The generated text is displayed in the output area.

This tool enables direct comparison between different trained models to evaluate how dataset size and number of epochs influence text generation quality.

In [ ]:
from transformers import EncoderDecoderModel, BertTokenizer
import torch
import ipywidgets as widgets
from IPython.display import display

model_paths = {
    ("30k", "EPOCH1"): "NahlaAboromi/shakespeare-bert-30k-epoch1",
    ("30k", "EPOCH2"): "NahlaAboromi/shakespeare-bert-30k-epoch2",
    ("30k", "EPOCH3"): "NahlaAboromi/shakespeare-bert-30k-epoch3",
    ("100K", "EPOCH1"): "NahlaAboromi/shakespeare-bert-100k-epoch1",
    ("100K", "EPOCH2"): "NahlaAboromi/shakespeare-bert-100k-epoch2",
    ("100K", "EPOCH3"): "NahlaAboromi/shakespeare-bert-100k-epoch3",
}

# פעולה
def generate_text(b):
    output_area.clear_output()
    with output_area:
        selected_key = (k_selector.value, epoch_selector.value)
        if selected_key not in model_paths:
            print("⚠️ No model found for this selection.")
            return

        folder = model_paths[selected_key]

        try:
            model = EncoderDecoderModel.from_pretrained(folder)
            tokenizer = BertTokenizer.from_pretrained(folder)
            model.eval()# Switches the model to evaluation mode (disables dropout & batch norm updates)
                        # This ensures stable and consistent behavior during text generation

            inputs = tokenizer(input_text.value, return_tensors="pt")
            output = model.generate(
                input_ids=inputs.input_ids,
                max_length=50,
                num_beams=5,
                no_repeat_ngram_size=2,
                early_stopping=True
            )

            generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
            print("📝 Generated Shakespearean Text:\n")
            print(generated_text)

        except Exception as e:
            print("❌ Error:", e)

# 🎛️ UI Elements
k_selector = widgets.Dropdown(
    options=["30k", "100K"],
    description="📁 Dataset:",
    style={'description_width': '100px'},
    layout=widgets.Layout(width='200px')
)

epoch_selector = widgets.Dropdown(
    options=["EPOCH1", "EPOCH2", "EPOCH3"],
    description="📅 Epoch:",
    style={'description_width': '100px'},
    layout=widgets.Layout(width='200px')
)

input_text = widgets.Textarea(
    value="When I consider everything that grows",
    description="✍️ Prompt:",
    layout=widgets.Layout(width="100%", height="120px"),
    style={'description_width': '100px'}
)

generate_button = widgets.Button(
    description="🎭 Generate Shakespearean Text",
    button_style="success",
    layout=widgets.Layout(width='100%', height='40px')
)

output_area = widgets.Output(layout={
    'border': '1px solid lightgray',
    'padding': '10px',
    'margin': '10px 0px 0px 0px',
    'max_height': '200px',
    'overflow_y': 'auto'
})

generate_button.on_click(generate_text)

# 🧩 UI Layout
title = widgets.HTML("<h2 style='color:#444; font-family:Arial;'>🎬 Shakespeare Text Generator</h2>")

selectors_row = widgets.HBox([k_selector, epoch_selector], layout=widgets.Layout(justify_content='flex-start', gap='20px'))

form_layout = widgets.VBox([
    title,
    selectors_row,
    input_text,
    generate_button,
    widgets.HTML("<b>📤 Output:</b>"),
    output_area
], layout=widgets.Layout(width='700px'))

# ✅ Show UI
display(form_layout)

